# 03. Data Preprocessing - Automobile Loan Default Prediction
**Module:** IT3051 – Fundamentals of Data Mining
**Objective:** Turn the raw, messy dataset into a clean, encoded, split, model-ready dataset, implementing every decision documented in `02_eda.ipynb`'s Section 11 summary.

---

### Recap of the plan (from `02_eda.ipynb` Section 11)
1. Setup - reload the raw data fresh (this notebook builds its own self-contained pipeline).
2. Fix data types (numeric-looking text columns).
3. Apply rule-based placeholder/sentinel fixes (`Employed_Days`, `XNA`, `"##"`, `Score_Source_2`, `Population_Region_Relative`), drop `ID`.
4. Stratified train/test split - **done before any step that learns a statistic from the data**, to avoid leakage.
5. Handle missing values (fit on training data only).
6. Outlier treatment (log-transform `Client_Income`).
7. Encode categorical variables (fit on training data only).
8. Feature scaling (fit on training data only).
9. Save processed train/test sets to `data/processed/`.
10. Summary of preprocessing decisions.

Rare-category grouping and new engineered features are deliberately left for `04_feature_engineering.ipynb`.


## 1. Setup

We reload the raw dataset fresh, independently of `02_eda.ipynb`'s exploration copy. This notebook is meant to be a self-contained, reusable pipeline — not a continuation of the EDA notebook's in-memory state — so that it can be re-run on its own (e.g. from the command line) to reproduce `data/processed/` from scratch.


In [1]:
import os
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 1000)

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
# Relative path handling: works whether launched from the workspace root or inside the notebooks folder
data_path = 'data/raw/Train_Dataset.csv' if os.path.exists('data/raw/Train_Dataset.csv') else '../data/raw/Train_Dataset.csv'

df = pd.read_csv(data_path, low_memory=False)

print(f"Dataset loaded successfully from: '{data_path}'")
print(f"Shape: {df.shape}")

Dataset loaded successfully from: '../data/raw/Train_Dataset.csv'
Shape: (121856, 40)


**Confirms:** shape should be `(121856, 40)`, matching both `01_data_understanding.ipynb` and `02_eda.ipynb` — this is the same untouched raw file, giving this notebook a clean, independent starting point to build the actual preprocessing pipeline from.


## 2. Fix Data Types

Convert numeric-looking text columns (found in `01_data_understanding.ipynb`) to real numbers; invalid entries become missing.


In [3]:
candidate_numerical_cols = [
    'Client_Income', 'Credit_Amount', 'Loan_Annuity', 'Population_Region_Relative',
    'Age_Days', 'Employed_Days', 'Registration_Days', 'ID_Days', 'Score_Source_3'
]

for col in candidate_numerical_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

print(df[candidate_numerical_cols].dtypes)

Client_Income                 float64
Credit_Amount                 float64
Loan_Annuity                  float64
Population_Region_Relative    float64
Age_Days                      float64
Employed_Days                 float64
Registration_Days             float64
ID_Days                       float64
Score_Source_3                float64
dtype: object


## 3. Rule-Based Placeholder/Sentinel Fixes

Fixed, non-data-dependent fixes found in `02_eda.ipynb`.


In [4]:
# Employed_Days sentinel -> flag + missing
df['Is_Retired_Or_Unemployed'] = (df['Employed_Days'] == 365243).astype(int)
df.loc[df['Employed_Days'] == 365243, 'Employed_Days'] = np.nan

# Disguised missing values -> NaN (Type_Organization's "XNA" is kept as-is, it's a valid category)
df.loc[df['Client_Gender'] == 'XNA', 'Client_Gender'] = np.nan
df.loc[df['Accompany_Client'] == '##', 'Accompany_Client'] = np.nan

# Corrupted numeric values -> NaN
df.loc[df['Score_Source_2'] > 1, 'Score_Source_2'] = np.nan
df.loc[df['Population_Region_Relative'] > 1, 'Population_Region_Relative'] = np.nan

# Drop non-predictive identifier
df = df.drop(columns=['ID'])

print("Is_Retired_Or_Unemployed counts:")
print(df['Is_Retired_Or_Unemployed'].value_counts())
print()
print("Shape after this step:", df.shape)

Is_Retired_Or_Unemployed counts:
Is_Retired_Or_Unemployed
0    100758
1     21098
Name: count, dtype: int64

Shape after this step: (121856, 40)


## 4. Stratified Train/Test Split

Split now, before any step that learns from the data, to avoid leakage. Stratify on `Default` to preserve the ~91.9%/8.1% class balance.


In [5]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df, test_size=0.2, stratify=df['Default'], random_state=42
)

print("train_df shape:", train_df.shape)
print("test_df shape:", test_df.shape)
print()
print("Default rate (train):", train_df['Default'].mean().round(4))
print("Default rate (test):", test_df['Default'].mean().round(4))

train_df shape: (97484, 40)
test_df shape: (24372, 40)

Default rate (train): 0.0808
Default rate (test): 0.0808


## 5. Handle Missing Values

Rule: numeric -> median (fit on train), with a `_was_missing` flag added when missing rate > 5% (preserves signal for high-missing columns). Categorical -> fill with `"Missing"` category (keeps informative missingness, e.g. `Client_Occupation`). Exception: `Own_House_Age` filled with 0 (no house owned), not median.


In [6]:
numeric_cols_missing = [c for c in train_df.select_dtypes(include=[np.number]).columns
                        if c != 'Default' and train_df[c].isna().sum() > 0]
categorical_cols_missing = [c for c in train_df.select_dtypes(include=['object', 'string']).columns
                            if train_df[c].isna().sum() > 0]

flag_threshold = 0.05
missing_value_fills = {}
flags_added = []

for col in numeric_cols_missing:
    if train_df[col].isna().mean() > flag_threshold:
        flag_col = col + '_was_missing'
        train_df[flag_col] = train_df[col].isna().astype(int)
        test_df[flag_col] = test_df[col].isna().astype(int)
        flags_added.append(flag_col)

    fill_value = 0 if col == 'Own_House_Age' else train_df[col].median()
    missing_value_fills[col] = fill_value
    train_df[col] = train_df[col].fillna(fill_value)
    test_df[col] = test_df[col].fillna(fill_value)

for col in categorical_cols_missing:
    train_df[col] = train_df[col].fillna('Missing')
    test_df[col] = test_df[col].fillna('Missing')

print("Missing-indicator flags added:", flags_added)
print()
print("Remaining missing values (train):", train_df.isnull().sum().sum())
print("Remaining missing values (test):", test_df.isnull().sum().sum())

Missing-indicator flags added: ['Employed_Days_was_missing', 'Own_House_Age_was_missing', 'Score_Source_1_was_missing', 'Score_Source_3_was_missing', 'Social_Circle_Default_was_missing', 'Credit_Bureau_was_missing']

Remaining missing values (train): 0
Remaining missing values (test): 0
